# Colab Setup — TFM Pose 6-DoF

**Este notebook configura TODO el entorno en Google Colab.**

## Estrategia Local vs Colab

| Tarea | Local (M1 Pro) | Colab (T4/A100) |
|-------|---------------|------------------|
| Desarrollo de codigo | Editor + git | Lento para editar |
| Docker / ROS 2 / CoppeliaSim | Docker Desktop | No soportado |
| Notebooks matematicos | Jupyter local | Tambien funciona |
| **Datasets BOP (30+ GB)** | Disco limitado | ~80 GB libres |
| **FoundationPose inferencia** | Necesita CUDA | T4/A100 GPU |
| **GDR-Net inferencia** | Necesita CUDA | T4/A100 GPU |
| **Diffusion Policy training** | Muy lento en CPU | GPU acelerado |
| **BOP Toolkit evaluacion** | CPU suficiente | Tambien funciona |
| Visualizacion / figuras TFM | Mejor local | Inline |

### Flujo de trabajo recomendado:
1. **Desarrollar** en local (VSCode + git)
2. **Push** a GitHub
3. **Pull** en Colab y ejecutar evaluaciones GPU
4. **Descargar** resultados (.json, figuras) a local
5. **Integrar** en memoria TFM

## 0. Configuracion Global

Todas las constantes compartidas se definen aqui para evitar `NameError` si se ejecutan celdas individuales.

In [ ]:
# ============================================================
# CONFIGURACION GLOBAL — Ejecutar SIEMPRE primero
# ============================================================
import os

REPO_URL  = "https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git"
REPO_DIR  = "/content/repo_tfm"
USE_GDRIVE = True   # Cambiar a False si no quieres persistencia en Drive

# Estrategia: zips en Drive (persistente), extraccion en /content/ (rapido)
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights" if USE_GDRIVE else f"{REPO_DIR}/weights"

print(f"REPO_DIR    = {REPO_DIR}")
print(f"WEIGHTS_DIR = {WEIGHTS_DIR}")
print(f"USE_GDRIVE  = {USE_GDRIVE}")
print(f"\nNota: Los datasets se extraen en /content/datasets/ (disco rapido)")

## 1. Verificar GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("\nVe a Runtime > Change runtime type > T4 GPU")

# Espacio en disco
!df -h /content | tail -1
!free -h | head -2

## 2. Clonar repositorio TFM

In [ ]:
import os

if os.path.exists(REPO_DIR):
    print("Repo ya existe, haciendo pull...")
    !cd {REPO_DIR} && git pull
else:
    print("Clonando repositorio...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git log --oneline -5

## 3. Instalar dependencias

In [ ]:
# ============================================================
# Dependencias — SOLO lo que realmente usa nuestro código
# ============================================================
# Colab ya tiene: torch, torchvision, numpy>=2, scipy, PIL, tqdm

# trimesh: carga modelos 3D CAD (.ply/.obj) en evaluator.py y foundation_pose.py
# diffusers: DDPM scheduler para Diffusion Policy
!pip install -q trimesh
!pip install -q diffusers accelerate transformers
!pip install -q scikit-learn

# NO instalamos open3d (fuerza numpy<2) — no se usa en src/
# NO instalamos pyrender — no se usa en src/
# NO instalamos bop_toolkit_lib (fuerza numpy<2) — nuestro metrics.py
#   ya implementa ADD, ADD-S, VSD, MSSD, MSPD

# Verificar que numpy sigue siendo >=2
import numpy as np
print(f"numpy: {np.__version__} (debe ser >=2.0)")
assert int(np.__version__.split('.')[0]) >= 2, "ERROR: numpy fue downgradeado!"

# Verificar imports del repo
import sys
sys.path.insert(0, REPO_DIR)
from src.utils.lie_groups import so3_exp, se3_exp
from src.utils.rotations import matrix_to_6d, sixd_to_matrix
from src.utils.metrics import compute_add, compute_adds
from src.planning.diffusion_policy import DiffusionGraspPlanner
from src.pipeline import BinPickingPipeline, PipelineConfig
print("\nTodos los imports OK")

## 4. Descargar Datasets BOP

**IMPORTANTE — Estrategia de almacenamiento:**
- Los zips se guardan en **Google Drive** (persistencia entre sesiones)
- La extracción se hace en **disco local `/content/`** (rápido, ~80 GB libres)
- Cada vez que abres Colab, solo extraes (no re-descargas)

Esto evita el problema de I/O lento en Drive al extraer miles de archivos.

In [ ]:
# ============================================================
# Montar Google Drive + Crear estructura
# ============================================================
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Zips en Drive (persistente), extraccion en disco local (rapido)
DRIVE_ZIPS = "/content/drive/MyDrive/TFM/datasets_zips"
LOCAL_DATA = "/content/datasets"   # Disco local rapido

os.makedirs(DRIVE_ZIPS, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

# Symlink para que el codigo del repo encuentre los datos
REPO_DATA = f"{REPO_DIR}/data/datasets"
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
os.symlink(LOCAL_DATA, REPO_DATA)
print(f"Symlink: {REPO_DATA} -> {LOCAL_DATA}")
print(f"Zips persistentes en Drive: {DRIVE_ZIPS}")
print(f"Datos extraidos en disco local: {LOCAL_DATA}")

In [ ]:
# ============================================================
# Funcion de descarga + extraccion
# ============================================================
import time

HF_BASE = "https://huggingface.co/datasets/bop-benchmark"

def download_and_extract(dataset, filename, url):
    """Descarga zip a Drive (si no existe) y extrae a disco local."""
    zip_on_drive = f"{DRIVE_ZIPS}/{filename}"
    extract_dir = f"{LOCAL_DATA}/{dataset}"
    os.makedirs(extract_dir, exist_ok=True)

    # Paso 1: Descargar zip a Drive (solo si no existe)
    if os.path.exists(zip_on_drive):
        size_mb = os.path.getsize(zip_on_drive) / 1e6
        print(f"  [CACHE] {filename} ya en Drive ({size_mb:.0f} MB)")
    else:
        print(f"  [DOWN] Descargando {filename}...")
        !wget -q --show-progress -c -O "{zip_on_drive}" "{url}"

    # Paso 2: Extraer a disco local (siempre rapido, ~2 min para 8 GB)
    print(f"  [UNZIP] Extrayendo a disco local...")
    t0 = time.time()
    !unzip -qo "{zip_on_drive}" -d "{extract_dir}"
    elapsed = time.time() - t0
    print(f"  [OK] {filename} extraido en {elapsed:.0f}s")

print("Funcion download_and_extract lista.")

In [ ]:
# ============================================================
# YCB-Video — 21 objetos domesticos
# Total: ~11 GB (base 15K + models 500M + test ~10G)
# ============================================================
print("=== YCB-Video ===")
download_and_extract("ycbv", "ycbv_base.zip",
    f"{HF_BASE}/ycbv/resolve/main/ycbv_base.zip")
download_and_extract("ycbv", "ycbv_models.zip",
    f"{HF_BASE}/ycbv/resolve/main/ycbv_models.zip")
download_and_extract("ycbv", "ycbv_test_all.zip",
    f"{HF_BASE}/ycbv/resolve/main/ycbv_test_all.zip")

# Reorganizar: BOP zips extraen en subcarpeta ycbv/, aplanar al nivel raiz
YCBV = f"{LOCAL_DATA}/ycbv"
# cp -r para copiar directorios completos (test/, models/, etc.)
!cp -r {YCBV}/ycbv/* {YCBV}/ 2>/dev/null; true
!cp {YCBV}/camera_uw.json {YCBV}/camera.json 2>/dev/null; true

# Verificar que test/ tiene imagenes RGB
print("\n=== YCB-V verificacion ===")
!ls {YCBV}/
first_scene=$(ls {YCBV}/test/ 2>/dev/null | head -1)
if [ -n "$first_scene" ]; then
    echo "Primera escena: $first_scene"
    echo "Contenido rgb/:"
    ls {YCBV}/test/$first_scene/rgb/ 2>/dev/null | head -3
    echo "Total imagenes:"
    ls {YCBV}/test/$first_scene/rgb/ 2>/dev/null | wc -l
else
    echo "ERROR: No hay escenas en test/"
fi

In [ ]:
# ============================================================
# T-LESS — 30 objetos industriales sin textura
# Total: ~8 GB (base 48K + models 32M + test 7.76G)
# ============================================================
print("=== T-LESS ===")
download_and_extract("tless", "tless_base.zip",
    f"{HF_BASE}/tless/resolve/main/tless_base.zip")
download_and_extract("tless", "tless_models.zip",
    f"{HF_BASE}/tless/resolve/main/tless_models.zip")
download_and_extract("tless", "tless_test_primesense_all.zip",
    f"{HF_BASE}/tless/resolve/main/tless_test_primesense_all.zip")

# Reorganizar: BOP extrae en subcarpetas, aplanar al nivel raiz
TLESS = f"{LOCAL_DATA}/tless"
!cp -r {TLESS}/tless/* {TLESS}/ 2>/dev/null; true
!ln -sf models_cad {TLESS}/models 2>/dev/null; true
!cp {TLESS}/camera_primesense.json {TLESS}/camera.json 2>/dev/null; true

# Verificar que test_primesense/ tiene imagenes RGB
print("\n=== T-LESS verificacion ===")
!ls {TLESS}/
first_scene=$(ls {TLESS}/test_primesense/ 2>/dev/null | head -1)
if [ -n "$first_scene" ]; then
    echo "Primera escena: $first_scene"
    echo "Contenido rgb/:"
    ls {TLESS}/test_primesense/$first_scene/rgb/ 2>/dev/null | head -3
    echo "Total imagenes:"
    ls {TLESS}/test_primesense/$first_scene/rgb/ 2>/dev/null | wc -l
else
    echo "ERROR: No hay escenas en test_primesense/"
fi

In [ ]:
# Resumen de espacio
print("=== Espacio en disco ===")
!df -h /content | tail -1
print("\n=== Datasets (disco local) ===")
!du -sh {LOCAL_DATA}/*/ 2>/dev/null || echo "No hay datos extraidos aun"
print(f"\n=== Zips en Drive ===")
!du -sh {DRIVE_ZIPS}/ 2>/dev/null || echo "No hay zips en Drive"
print(f"\n=== Symlink verificado ===")
!ls -la {REPO_DIR}/data/datasets/

## 5. Verificar Dataset Loader

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from src.utils.dataset_loader import BOPDataset
import matplotlib.pyplot as plt
import numpy as np

# Cargar YCB-V
try:
    ycbv = BOPDataset(f"{REPO_DIR}/data/datasets/ycbv", split="test")
    scenes = ycbv.get_scene_ids()
    print(f"YCB-V: {len(scenes)} escenas de test")

    # Usar get_image_ids() para obtener el primer ID real (no asumir 0)
    img_ids = ycbv.get_image_ids(scenes[0])
    print(f"  Escena {scenes[0]}: {len(img_ids)} imagenes, IDs {img_ids[0]}..{img_ids[-1]}")
    
    sample = ycbv.load_sample(scenes[0], img_ids[0])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(sample['rgb']); ax1.set_title(f'YCB-V Scene {scenes[0]} - RGB')
    ax2.imshow(sample['depth'], cmap='viridis'); ax2.set_title('Depth')
    plt.tight_layout(); plt.show()
    print(f"  RGB shape: {sample['rgb'].shape}")
    print(f"  Depth shape: {sample['depth'].shape}")
    print(f"  Camera K:\n{sample['cam_K']}")
except Exception as e:
    print(f"YCB-V no cargado: {e}")

# Cargar T-LESS
try:
    tless = BOPDataset(f"{REPO_DIR}/data/datasets/tless", split="test_primesense")
    scenes = tless.get_scene_ids()
    print(f"\nT-LESS: {len(scenes)} escenas de test")

    img_ids = tless.get_image_ids(scenes[0])
    print(f"  Escena {scenes[0]}: {len(img_ids)} imagenes, IDs {img_ids[0]}..{img_ids[-1]}")
    
    sample = tless.load_sample(scenes[0], img_ids[0])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(sample['rgb']); ax1.set_title(f'T-LESS Scene {scenes[0]} - RGB')
    ax2.imshow(sample['depth'], cmap='viridis'); ax2.set_title('Depth')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"T-LESS no cargado: {e}")

## 6. Descargar Pesos Pre-entrenados

In [ ]:
# ============================================================
# FoundationPose weights (NVIDIA)
# ============================================================
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if USE_GDRIVE:
    # Symlink para que el repo encuentre los pesos
    repo_weights = f"{REPO_DIR}/weights"
    if os.path.islink(repo_weights): os.unlink(repo_weights)
    elif os.path.isdir(repo_weights): 
        import shutil; shutil.rmtree(repo_weights)
    os.symlink(WEIGHTS_DIR, repo_weights)

FP_WEIGHTS = f"{WEIGHTS_DIR}/foundationpose"
os.makedirs(FP_WEIGHTS, exist_ok=True)

# FoundationPose usa modelos de NeRF + pose refiner
# Los pesos oficiales estan en el repo de NVIDIA
print("FoundationPose weights:")
print("Los pesos se descargan automaticamente al ejecutar el modelo.")
print(f"Directorio: {FP_WEIGHTS}")
print("\nPara descarga manual:")
print("  https://github.com/NVlabs/FoundationPose#pre-trained-weights")

In [ ]:
# ============================================================
# GDR-Net++ weights (BOP 2022 winner)
# ============================================================
GDRNET_WEIGHTS = f"{WEIGHTS_DIR}/gdrnet"
os.makedirs(GDRNET_WEIGHTS, exist_ok=True)

print("GDR-Net++ weights:")
print("  Pre-entrenados disponibles en:")
print("  https://github.com/shanice-l/gdrnpp_bop2022#model-zoo")
print(f"\nDirectorio: {GDRNET_WEIGHTS}")

## 7. Quick Sanity Check — GPU + Nuestro Codigo

In [ ]:
import torch
import numpy as np
import sys; sys.path.insert(0, REPO_DIR)
from src.utils.lie_groups import so3_exp, pose_from_Rt, geodesic_distance_SE3
from src.planning.diffusion_policy import DiffusionGraspPlanner, ConditionalUNet1D

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Test 1: UNet1D en GPU
model = ConditionalUNet1D(action_dim=7, horizon=16, cond_dim=64).to(device)
x = torch.randn(4, 16, 7, device=device)  # batch=4
t = torch.randint(0, 100, (4,), device=device)
c = torch.randn(4, 64, device=device)

with torch.no_grad():
    out = model(x, t, c)
print(f"UNet1D forward pass: {out.shape} on {device}")

# Test 2: Grasp planner en GPU
planner = DiffusionGraspPlanner(device=device)
R = so3_exp(np.array([0.1, -0.2, 0.3]))
T = pose_from_Rt(R, np.array([0.15, -0.08, 0.45]))
traj = planner.plan_grasp_heuristic(T)
print(f"Heuristic grasp: {traj.shape}")

# Test 3: Benchmark GPU throughput
if device == "cuda":
    import time
    batch = torch.randn(64, 16, 7, device=device)
    ts = torch.randint(0, 100, (64,), device=device)
    conds = torch.randn(64, 64, device=device)
    
    # Warmup
    for _ in range(10): model(batch, ts, conds)
    torch.cuda.synchronize()
    
    t0 = time.time()
    for _ in range(100): model(batch, ts, conds)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    print(f"GPU throughput: {100*64/elapsed:.0f} samples/s ({elapsed/100*1000:.1f} ms/batch)")

print(f"\nTodo funciona correctamente en Colab con {device.upper()}!")

## 8. Guardar resultados a Google Drive

Utility para guardar resultados de experimentos

In [ ]:
import json
from datetime import datetime

def save_experiment(name, results, figures=None):
    """Guardar resultados de experimento a Drive.
    
    Args:
        name: nombre del experimento (e.g. 'foundationpose_ycbv')
        results: dict con metricas
        figures: list of (fig, filename) tuples
    """
    exp_dir = f"{REPO_DIR}/experiments/{name}"
    if USE_GDRIVE:
        exp_dir = f"/content/drive/MyDrive/TFM/experiments/{name}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Guardar metricas
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    results['timestamp'] = timestamp
    results['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    
    with open(f"{exp_dir}/results_{timestamp}.json", 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Resultados guardados: {exp_dir}/results_{timestamp}.json")
    
    # Guardar figuras
    if figures:
        for fig, fname in figures:
            fig.savefig(f"{exp_dir}/{fname}", dpi=150, bbox_inches='tight')
            print(f"Figura: {exp_dir}/{fname}")
    
    return exp_dir

# Ejemplo
save_experiment('test_setup', {
    'status': 'Colab configurado correctamente',
    'pytorch': torch.__version__,
    'cuda': torch.cuda.is_available(),
})
print("\nSetup completo! Puedes continuar con los notebooks de evaluacion.")